# Epileptic Seizure Cartesian Benchmark (Colab)

This notebook runs the full staged Cartesian benchmark with all requested technologies:
- Preprocessing: Standard, MinMax, Robust, Quantile
- Reduction: none, PCA, LDA projection, SVD
- Selection: none, chi2, ANOVA, correlation, SFS, RFE, embedded L1, GA
- Classifiers: KNN, SVM, Decision Tree, Logistic Regression, LDA classifier, MLP/ANN
- Tracks: binary + multiclass
- CV: 3 folds

Expected unique combos: `4 x 4 x 8 x 6 x 2 = 1536`
Expected fold evaluations: `1536 x 3 = 4608`


In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn tabulate
print("Dependencies installed.")


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/adhamhaithameid/soft-computing-main-project.git"
REPO_DIR = Path("/content/soft-computing-main-project")

if (Path.cwd() / "src").exists():
    project_dir = Path.cwd()
elif (REPO_DIR / "src").exists():
    project_dir = REPO_DIR
else:
    subprocess.run(f"git clone --depth 1 {REPO_URL} {REPO_DIR}", shell=True, check=True)
    project_dir = REPO_DIR

os.chdir(project_dir)
print("Using project directory:", project_dir)
!ls -la


In [ ]:
# Config
SMOKE_TEST = False          # set True for quick validation run
MAX_ROWS = 120 if SMOKE_TEST else None
FRESH_RUN = True
CHECKPOINT_EVERY = 120
SEED = 42

PREPROCESSING_METHODS = ["standard", "minmax", "robust", "quantile"]
REDUCTION_METHODS = ["none", "pca", "lda_projection", "svd"]
SELECTION_METHODS = [
    "none", "filter_chi2", "filter_anova", "filter_correlation",
    "wrapper_sfs", "wrapper_rfe", "embedded_l1", "ga_selection"
]
CLASSIFIER_METHODS = ["knn", "svm", "decision_tree", "logistic_regression", "lda_classifier", "mlp_ann"]
TRACKS = ["binary", "multiclass"]
CV_SPLITS = 3

expected_combos = len(PREPROCESSING_METHODS) * len(REDUCTION_METHODS) * len(SELECTION_METHODS) * len(CLASSIFIER_METHODS) * len(TRACKS)
expected_fold_evals = expected_combos * CV_SPLITS
print("expected_combos:", expected_combos)
print("expected_fold_evals:", expected_fold_evals)


In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config.paths import (
    DATASET_CSV,
    DATA_PROCESSED_DIR,
    RESULTS_METRICS_DIR,
    RESULTS_TABLES_DIR,
    RESULTS_FIGURES_DIR,
    RESULTS_REPORTS_DIR,
)
from src.core import (
    CartesianSpec,
    RunnerIO,
    run_cartesian_benchmark,
    build_summary,
    save_comparisons,
    generate_cartesian_plots,
)

if not DATASET_CSV.exists():
    print(f"Dataset missing at {DATASET_CSV}, running fetch script...")
    import subprocess
    subprocess.run("python src/cli/fetch_data.py", shell=True, check=True)

df = pd.read_csv(DATASET_CSV)
unnamed = [c for c in df.columns if c.lower().startswith("unnamed")]
if unnamed:
    df = df.drop(columns=unnamed)

target_col = "y" if "y" in df.columns else df.columns[-1]
for col in df.columns:
    if col != target_col:
        df[col] = pd.to_numeric(df[col], errors="coerce")

X_df = df.drop(columns=[target_col]).copy()
X_df = X_df.fillna(X_df.median(numeric_only=True))
y_multi = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int).to_numpy()
y_binary = (y_multi == 1).astype(int)

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
X_df.to_csv(DATA_PROCESSED_DIR / "features_numeric.csv", index=False)
pd.DataFrame({"y_multiclass": y_multi, "y_binary": y_binary}).to_csv(DATA_PROCESSED_DIR / "targets.csv", index=False)

print("Loaded dataset shape:", X_df.shape)
print("Binary class counts:", pd.Series(y_binary).value_counts().sort_index().to_dict())
print("Multiclass class counts:", pd.Series(y_multi).value_counts().sort_index().to_dict())


In [ ]:
from scipy.stats import kurtosis, skew
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

desc = pd.DataFrame({
    "min": X_df.min(),
    "max": X_df.max(),
    "mean": X_df.mean(),
    "variance": X_df.var(),
    "std": X_df.std(),
    "skewness": X_df.apply(lambda s: skew(s.to_numpy(), bias=False)),
    "kurtosis": X_df.apply(lambda s: kurtosis(s.to_numpy(), bias=False)),
})
desc.to_csv(RESULTS_TABLES_DIR / "dataset_descriptive_stats.csv")

cov = X_df.cov()
corr = X_df.corr()
cov.to_csv(RESULTS_TABLES_DIR / "covariance_matrix.csv")
corr.to_csv(RESULTS_TABLES_DIR / "correlation_matrix.csv")

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(RESULTS_FIGURES_DIR / "correlation_heatmap.png", dpi=200)
plt.close()

print("Statistical analysis outputs saved.")


In [ ]:
spec = CartesianSpec(
    preprocessing=tuple(PREPROCESSING_METHODS),
    reduction=tuple(REDUCTION_METHODS),
    selection=tuple(SELECTION_METHODS),
    classifiers=tuple(CLASSIFIER_METHODS),
    tracks=tuple(TRACKS),
    cv_splits=CV_SPLITS,
)

metrics_csv = RESULTS_METRICS_DIR / "cartesian_metrics_all.csv"
manifest_json = RESULTS_METRICS_DIR / "cartesian_run_manifest.json"
if FRESH_RUN:
    metrics_csv.unlink(missing_ok=True)
    manifest_json.unlink(missing_ok=True)

io = RunnerIO(metrics_csv=metrics_csv, manifest_json=manifest_json, checkpoint_every=CHECKPOINT_EVERY)
metrics_df = run_cartesian_benchmark(
    X_df=X_df,
    y_binary=y_binary,
    y_multiclass=y_multi,
    spec=spec,
    io=io,
    resume=not FRESH_RUN,
    max_rows=MAX_ROWS,
)

ok = metrics_df[metrics_df["status"] == "ok"].copy()
summary = build_summary(ok).sort_values(["track", "accuracy"], ascending=[True, False])
summary.to_csv(RESULTS_TABLES_DIR / "cartesian_summary_by_combo.csv", index=False)

saved = save_comparisons(summary, RESULTS_TABLES_DIR, RESULTS_REPORTS_DIR)
plots = generate_cartesian_plots(metrics_df, summary, RESULTS_FIGURES_DIR, X_df, y_binary, CV_SPLITS)

print("Run completed")
print("metrics rows:", len(metrics_df))
print("ok rows:", int((metrics_df["status"] == "ok").sum()))
print("failed rows:", int((metrics_df["status"] != "ok").sum()))
print("plots generated:", len(plots))
print("comparison report:", saved["report"])


In [ ]:
import json

manifest = json.loads((RESULTS_METRICS_DIR / "cartesian_run_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))

if MAX_ROWS is None:
    assert manifest["expected_combos"] == 1536, manifest["expected_combos"]
    assert manifest["expected_fold_evals"] == 4608, manifest["expected_fold_evals"]
    assert len(metrics_df) == 4608, len(metrics_df)
else:
    print("Smoke mode active; strict full-count assertions skipped.")


In [ ]:
display(summary[summary["track"] == "binary"].head(10))
display(summary[summary["track"] == "multiclass"].head(10))


In [ ]:
import os
figs = sorted([p.name for p in RESULTS_FIGURES_DIR.glob("cartesian_*.png")])
print("Generated figure files:")
for f in figs:
    print("-", f)


In [ ]:
!zip -rq colab_outputs.zip results paper/draft data/processed
print("Created colab_outputs.zip")
